# Fase 1 — Adquisición y validación de datos
## Proyecto integrador "La hora dorada"

La **implementación** de la Fase 1 vive en módulos `.py` reutilizables
(como pide el enunciado):

| Paso | Módulo | Qué hace |
|---|---|---|
| 1. Adquisición / carga | [`src/acquisition.py`](src/acquisition.py) | resuelve cada fuente (ya existe / caché `_data/` / descarga), controla el encoding (`chardet`), registra en `data/raw/download_log.json` |
| 2-4. Limpieza, validación, filtrado, almacenamiento | [`src/validation.py`](src/validation.py) | estandariza categorías (`II-1`) y estados (`ACTIVO`); valida coordenadas (vacías/cero, **signo de hemisferio**, **intercambio lat/lon**, fuera de Perú); duplicados; punto-fuera-de-su-distrito → `logs/quality_report.csv`; recorta a los 3 departamentos de `config.md`; guarda GeoPackage en `data/processed/` |
| 5. Exploración | [`src/mapas.py`](src/mapas.py) | un mapa interactivo Folium por departamento → `data/outputs/mapa_acceso_<depto>.html` |

Este notebook solo **ejecuta esos módulos y muestra los mapas en línea**.
Equivale a:

```bash
python src/acquisition.py
python src/validation.py
```


## Paso 1 — Adquisición / carga


In [ ]:
from src.acquisition import cargar_config, run as adquirir

CFG = cargar_config()
adquirir(CFG)                       # copia de _data/ o descarga; llena data/raw/


## Pasos 2-4 — Limpieza, validación, filtrado de ámbito y almacenamiento


In [ ]:
from src.validation import run as validar

resultado = validar(CFG, generar_mapas=True)   # data/processed/*.gpkg + logs/quality_report.csv + mapas
import pandas as pd
pd.DataFrame(resultado["resumen"])


### Informe de Calidad de Datos (`logs/quality_report.csv`)


In [ ]:
import pandas as pd
pd.read_csv("logs/quality_report.csv")


## Paso 5 — Exploración con Folium

Un mapa interactivo por departamento (fondo claro, `MarkerCluster` para la
demanda, ficha HTML en cada establecimiento resolutivo, `LayerControl`).
Los HTML ya se guardaron en `data/outputs/`; aquí se muestran en línea.


In [ ]:
import geopandas as gpd
from IPython.display import display
from src.acquisition import ruta, codigos_inei
from src.validation import filtrar_ambito
from src.mapas import mapa_departamento

proc = ruta(CFG["rutas"]["processed"])
oferta = gpd.read_file(proc / "renipress_nacional.gpkg")
demanda = gpd.read_file(proc / "centros_poblados_nacional.gpkg")
rd = ruta(CFG["fuentes"]["limites_administrativos"]["capas"]["distrito"]["archivo_local"])
distritos = gpd.read_file(rd) if rd.exists() else None
oferta_amb, demanda_amb, ub_cp = filtrar_ambito(oferta, demanda, CFG)

for dep, cod in codigos_inei(CFG).items():
    print(f"\n=== {dep} ===")
    display(mapa_departamento(dep, cod, oferta_amb, demanda_amb, distritos, ub_cp))


## Próximo paso — Fase 2 (enrutamiento)

`python src/routing.py --all` construye el grafo vial desde
`data/raw/peru-latest.osm.pbf` y calcula el tiempo real por carretera de
cada centro poblado al establecimiento **resolutivo** más cercano.
